In [ ]:
import os
import numpy as np
import pandas as pd


import pickle
from pathlib import Path

In [ ]:
results = "/home/jupyter/workspaces/infectiousdiseasephewas2/results/2026-01-27_curate_controls_for_PheWAS_study"
results1 = "/home/jupyter/workspaces/infectiousdiseasephewas2/results/2025-11-13_get_conditions_of_cohorts"

data = "/home/jupyter/workspaces/infectiousdiseasephewas2/data/2026-01-27_curate_controls_for_PheWAS_study"
data2 ="/home/jupyter/workspaces/infectiousdiseasephewas2/data/2025-11-13_get_cohort_statistic"

scratch="/home/jupyter/workspaces/infectiousdiseasephewas2/scratch/2026-01-27_curate_controls_for_PheWAS_study"

#!mkdir ${scratch}

In [ ]:
#import and merge phecode mapping files

In [ ]:
def import_map_files():
   
    
    
    phe_icd= pd.read_csv(f"{data}/expanded_phecode.csv")
    
    phe = pd.read_csv(f"{data}/phecode_info.csv")
    
    phex = pd.read_csv(f"{data}/phecodex_info.csv")
    
    phex = phex.rename({"phecodex": "phecode"}, axis="columns")
    
    phex_icd = pd.read_csv(f"{data}/updated_phecodex_map.csv")
    
    
    return phe, phe_icd, phex, phex_icd
    

In [ ]:
def merge_phecodes_with_phenotype(phe, phe_icd):
    
    final = pd.merge(phe, phe_icd, how = "inner")
    
    return final

In [ ]:
#import cohort dataframe and wrangle into cohorts

In [ ]:
def get_data_pkl(path, filename):

    # …later, in any notebook in the same workspace…
    out_file = Path(f'{path}/{filename}')

    # Reload:
    with open(out_file, 'rb') as f:
        cohort_data_dict = pickle.load(f)

    #print("Reloaded keys:", list(cohort_data_dict.keys()))
    
    return cohort_data_dict

In [ ]:
def get_cond_cohort_df_dict(all_cond_df): 
    print("Grouping DataFrame into a dictionary...")

    # Define the columns you want to use for your composite key
    key_columns = ['condition_concept_id', 'standard_concept_name']

    # Use a dictionary comprehension with groupby to create the dictionary
    # - The 'key' will be a tuple: (condition_concept_id, standard_concept_name)
    # - The 'group_df' will be the DataFrame containing all rows for that key
    concept_groups_dict = {
        key: group_df 
        for key, group_df in all_cond_df.groupby(key_columns)
    }

    print(f"Successfully created a dictionary with {len(concept_groups_dict)} unique (ID, Name) keys.")
    
    return concept_groups_dict

In [ ]:
def filter_cond_df_dict(cohort_dict):
    
    
    master_df_of_final_cohorts = pd.read_csv(f"{results1}/viral_disease_condition_cohorts.csv")

    master_key_cols = ['condition_concept_id', 'standard_concept_name']

    # Create a set of tuples (key1, key2) from the master DataFrame.
    # This set will act as our "allow list".
    valid_keys_set = set(
        master_df_of_final_cohorts[master_key_cols].itertuples(index=False, name=None)
    )
    
    # 'concept_groups_dict' is the dictionary you created in the previous step
    # 'valid_keys_set' is the set we just created

    filtered_concept_dict = {
        key: group_df 
        for key, group_df in cohort_dict.items() 
        if key in valid_keys_set
    }

    print(f"Original dictionary had {len(cohort_dict)} items.")
    print(f"Filtered dictionary now has {len(filtered_concept_dict)} items.")

    # You can now work with your new, smaller dictionary
    # print(list(filtered_concept_dict.keys())[:5])
    
    return filtered_concept_dict

In [ ]:
#count the proportion of unqiue samples coded with an ICD9/10 code atleast 1 in each of my cohorts

In [ ]:
def get_icd_vocab_count_per_cohort(filtered_cohort_dict):
    
    icd_vocabs = ["ICD9CM", "ICD10CM"]

    results = []

    for key, df in filtered_cohort_dict.items():
        total_individuals = df["person_id"].nunique()
        mask_icd = df["source_vocabulary"].isin(icd_vocabs)
        individuals_with_icd = df.loc[mask_icd, "person_id"].nunique()
        prop_with_icd = individuals_with_icd / total_individuals if total_individuals > 0 else float("nan")

        results.append({
            "key": key,
            "total_individuals": total_individuals,
            "with_icd": individuals_with_icd,
            "prop_with_icd": prop_with_icd,
        })

    summary = pd.DataFrame(results)
    #summary.to_csv("cohort_source_vocab_count.csv")

    return summary


In [ ]:
#create a cohort concept_name/id to IDC9/10 code mapping file from filtered cohohort dictionary

In [ ]:
def get_cohort_concept_id_to_ICD_mapping(raw_cohort_df):   
    
    master_df_of_final_cohorts = pd.read_csv(f"{results1}/viral_disease_condition_cohorts.csv")
    
    # columns we want in the final df
    cols = ['condition_concept_id', 'standard_concept_name', "source_concept_code", "source_vocabulary"]

    # keep only ICD9/10 rows, then select the 3 columns and drop duplicates
    icd_df = (
        raw_cohort_df[raw_cohort_df["source_vocabulary"].isin(["ICD9CM", "ICD10CM"])]
        [cols]
        .dropna(subset=["source_concept_code"])   
        .drop_duplicates()
    )
    
    
    master_cohort_list = master_df_of_final_cohorts["condition_concept_id"]
    final = icd_df[icd_df["condition_concept_id"].isin(master_cohort_list)]
    
    
    return final


In [ ]:
#get percent of ICD overlap of my cohorts to phecodes dataframe

# ---------------------------
# 1. Load + filter to ICD9/10
# ---------------------------


icd_vocabs = ["ICD9CM", "ICD10CM"]


# cond columns: condition_concept_id, standard_concept_name, source_concept_code, source_vocabulary
# phe  columns: phecode, description, group, vocabulary_id, concept_code

# -----------------------------------------
# 2. Unique condition–ICD and phecode–ICD
# -----------------------------------------

cond_codes = cohort_ICD_map[
    ["condition_concept_id", "standard_concept_name", "source_concept_code", "source_vocabulary"]]


# combine phecode + phecodeX
phe_all = pd.concat([phe_map, phex_map], ignore_index=True)


phe_codes = phe_all[
    ["phecode", "description", "group", "concept_code", "vocabulary_id"]
].drop_duplicates()


# -------------------------------------------------
# 3. Denominator: how many ICD codes per condition
# -------------------------------------------------

cohort_icd_count = (
    cond_codes
    .groupby("condition_concept_id")["source_concept_code"]
    .nunique()
    .rename("n_my_icds")
    .reset_index()
)

# --------------------------------------------
# 4. Overlaps: join on code + vocabulary
# --------------------------------------------

overlap = cond_codes.merge(
    phe_codes,
    left_on=["source_concept_code", "source_vocabulary"],
    right_on=["concept_code", "vocabulary_id"],
    how="inner",
)

# Count overlapping ICD codes for each (condition, phecode)
overlap_counts = (
    overlap
    .groupby(["condition_concept_id", "phecode"])["source_concept_code"]
    .nunique()
    .rename("n_overlap")
    .reset_index()
)


# Add denominator and compute fraction
overlap_counts = overlap_counts.merge(
    cohort_icd_count,
    on="condition_concept_id",
    how="left"
)

overlap_counts["frac_overlap"] = overlap_counts["n_overlap"] / overlap_counts["n_my_icds"]

# --------------------------------------------
# 5. (Optional) attach names / descriptions
# --------------------------------------------

overlap_counts = overlap_counts.merge(
    cohort_ICD_map[["condition_concept_id", "standard_concept_name"]].drop_duplicates(),
    on="condition_concept_id",
    how="left"
)

overlap_counts = overlap_counts.merge(
    phe_all[["phecode", "description"]].drop_duplicates(),
    on="phecode",
    how="left"
)

# nice column order
overlap_counts = overlap_counts[
    [
        "condition_concept_id",
        "standard_concept_name",
        "phecode",
        "description",
        "n_my_icds",
        "n_overlap",
        "frac_overlap",
    ]
]


#1. Donor → ICD codes (File 1 → “dictionary 1” in pandas)
##Turn the wide donor × ICD matrix into a long table of (donor_id, concept_code).


# ---------- File 1 ----------
wide1 = pd.read_csv(f"{data}/mcc2_phecode_table.csv", dtype={"person_id": str})

# melt into long format
donor_codes1 = wide1.melt(
    id_vars="person_id",
    var_name="concept_code",
    value_name="has_code"
)

wide2 = pd.read_csv(f"{data}/mcc2_phecodex_table.csv", dtype={"person_id": str})

# melt into long format
donor_codes2 = wide2.melt(
    id_vars="person_id",
    var_name="concept_code",
    value_name="has_code"
)


donor_codes = pd.concat([donor_codes1, donor_codes2], ignore_index=True)


# keep only codes that are present (True / 1)
donor_codes = donor_codes[donor_codes["has_code"].astype(bool)].drop(columns="has_code")

donor_codes["concept_code"] = donor_codes["concept_code"].astype(str)

# donor_codes is the pandas version of "dict1: ICD -> list of donors"
# columns: donor_id, concept_code


#
'''
2. Get all donors that have B’s ICDs (Phenotype B donors)

This replaces:

Phenotype B donors = empty set()
For Phenotype B ICD in Phenotype B ICDs:
  Phenotype B donors.add(dictionary 1[Phenotype B ICD])

We just join max_exploded to donor_codes on concept_code:
'''

# join B ICDs to donors
b_donors = overlap.merge(
    donor_codes,
    on="concept_code",
    how="left"
)

# drop ICDs that no donor has
b_donors = b_donors.dropna(subset=["person_id"])

# one row per (A, B, donor)
b_donors_unique = b_donors[["condition_concept_id", "phecode", "person_id"]].drop_duplicates()

# count donors per (A,B) for Phenotype B side
b_counts = (
    b_donors_unique
    .groupby(["condition_concept_id", "phecode"])["person_id"]
    .nunique()
    .rename("n_donors_B")
    .reset_index()
)


#STEP 3: build a_donors_unique from the dict ----------

# phenotype_dict: {(condition_concept_id, name): df, ...}

a_donors_unique = pd.concat(
    [
        df[["person_id"]]
        .drop_duplicates()
        .assign(condition_concept_id=cond_id)
        for (cond_id, name), df in filtered_cohort_dict.items()
    ],
    ignore_index=True,
)



# same as before: count donors per phenotype A
a_counts = (
    a_donors_unique
    .groupby("condition_concept_id")["person_id"]
    .nunique()
    .rename("n_donors_A")
    .reset_index()
)

# make merge keys the same dtype on both sides
for col in ["condition_concept_id", "person_id"]:
    a_donors_unique[col] = a_donors_unique[col].astype("Int64")
    b_donors_unique[col] = b_donors_unique[col].astype("Int64")

overlap_donors = a_donors_unique.merge(
    b_donors_unique,
    on=["condition_concept_id", "person_id"],
    how="inner"
)
# columns: condition_concept_id, donor_id, phecode

overlap_counts_donors = (
    overlap_donors
    .groupby(["condition_concept_id", "phecode"])["person_id"]
    .nunique()
    .rename("n_overlap_donors")
    .reset_index()
)



# start from max_overlap so we keep exactly those (A,B) pairs from File 3
result = max_overlap[["condition_concept_id", "phecode"]].drop_duplicates()

# add counts and overlaps
result = result.merge(a_counts, on="condition_concept_id", how="left")
result = result.merge(b_counts, on=["condition_concept_id", "phecode"], how="left")
result = result.merge(overlap_counts_donors, on=["condition_concept_id", "phecode"], how="left")

# fill missing counts/overlaps with 0
for col in ["n_donors_A", "n_donors_B", "n_overlap_donors"]:
    result[col] = result[col].fillna(0).astype(int)

# fractions (avoid division by zero)
result["frac_A"] = result["n_overlap_donors"] / result["n_donors_A"].replace(0, pd.NA)
result["frac_B"] = result["n_overlap_donors"] / result["n_donors_B"].replace(0, pd.NA)

# keep the columns you said you want
out = result[[
    "condition_concept_id",   # Phenotype A
    "phecode",                # Phenotype B
    "n_overlap_donors",       # Overlap
    "frac_A",                 # Phenotype A fraction
    "frac_B",                 # Phenotype B fraction
]]


result = result.merge(
    max_overlap[["condition_concept_id", "phecode", "frac_overlap"]],
    on=["condition_concept_id", "phecode"],
    how="left"
)



In [ ]:
#get percent of ICD overlap of my cohorts to phecodes dataframe

# ---------------------------
# 1. Load + filter to ICD9/10
# ---------------------------


icd_vocabs = ["ICD9CM", "ICD10CM"]


# cond columns: condition_concept_id, standard_concept_name, source_concept_code, source_vocabulary
# phe  columns: phecode, description, group, vocabulary_id, concept_code

# -----------------------------------------
# 2. Unique condition–ICD and phecode–ICD
# -----------------------------------------

cond_codes = cohort_ICD_map[
    ["condition_concept_id", "standard_concept_name", "source_concept_code", "source_vocabulary"]]


# combine phecode + phecodeX
phe_all = pd.concat([phe_map, phex_map], ignore_index=True)


phe_codes = phe_all[
    ["phecode", "description", "group", "concept_code", "vocabulary_id"]
].drop_duplicates()


# -------------------------------------------------
# 3. Denominator: how many ICD codes per condition
# -------------------------------------------------

cohort_icd_count = (
    cond_codes
    .groupby("condition_concept_id")["source_concept_code"]
    .nunique()
    .rename("n_my_icds")
    .reset_index()
)

# --------------------------------------------
# 4. Overlaps: join on code + vocabulary
# --------------------------------------------

overlap = cond_codes.merge(
    phe_codes,
    left_on=["source_concept_code", "source_vocabulary"],
    right_on=["concept_code", "vocabulary_id"],
    how="inner",
)

# Count overlapping ICD codes for each (condition, phecode)
overlap_counts = (
    overlap
    .groupby(["condition_concept_id", "phecode"])["source_concept_code"]
    .nunique()
    .rename("n_overlap")
    .reset_index()
)


# Add denominator and compute fraction
overlap_counts = overlap_counts.merge(
    cohort_icd_count,
    on="condition_concept_id",
    how="left"
)

overlap_counts["frac_overlap"] = overlap_counts["n_overlap"] / overlap_counts["n_my_icds"]

# --------------------------------------------
# 5. (Optional) attach names / descriptions
# --------------------------------------------

overlap_counts = overlap_counts.merge(
    cohort_ICD_map[["condition_concept_id", "standard_concept_name"]].drop_duplicates(),
    on="condition_concept_id",
    how="left"
)

overlap_counts = overlap_counts.merge(
    phe_all[["phecode", "description"]].drop_duplicates(),
    on="phecode",
    how="left"
)

# nice column order
overlap_counts = overlap_counts[
    [
        "condition_concept_id",
        "standard_concept_name",
        "phecode",
        "description",
        "n_my_icds",
        "n_overlap",
        "frac_overlap",
    ]
]




In [ ]:

#1. Donor → ICD codes (File 1 → “dictionary 1” in pandas)
##Turn the wide donor × ICD matrix into a long table of (donor_id, concept_code).


# ---------- File 1 ----------
wide1 = pd.read_csv(f"{data}/mcc2_phecode_table.csv", dtype={"person_id": str})

# melt into long format
donor_codes1 = wide1.melt(
    id_vars="person_id",
    var_name="concept_code",
    value_name="has_code"
)




In [ ]:

#1. Donor → ICD codes (File 1 → “dictionary 1” in pandas)
##Turn the wide donor × ICD matrix into a long table of (donor_id, concept_code).


wide2 = pd.read_csv(f"{data}/mcc2_phecodex_table.csv", dtype={"person_id": str})

# melt into long format
donor_codes2 = wide2.melt(
    id_vars="person_id",
    var_name="concept_code",
    value_name="has_code"
)


donor_codes = pd.concat([donor_codes1, donor_codes2], ignore_index=True)

# keep only codes that are present (True / 1)
donor_codes = donor_codes[donor_codes["has_code"].astype(bool)].drop(columns="has_code")

donor_codes["concept_code"] = donor_codes["concept_code"].astype(str)

# donor_codes is the pandas version of "dict1: ICD -> list of donors"
# columns: donor_id, concept_code




In [ ]:

'''
2. Get all donors that have B’s ICDs (Phenotype B donors)

This replaces:

Phenotype B donors = empty set()
For Phenotype B ICD in Phenotype B ICDs:
  Phenotype B donors.add(dictionary 1[Phenotype B ICD])

We just join max_exploded to donor_codes on concept_code:
'''

# join B ICDs to donors
b_donors = overlap.merge(
    donor_codes,
    on="concept_code",
    how="left"
)

# drop ICDs that no donor has
b_donors = b_donors.dropna(subset=["person_id"])

# one row per (A, B, donor)
b_donors_unique = b_donors[["condition_concept_id", "phecode", "person_id"]].drop_duplicates()

# count donors per (A,B) for Phenotype B side
b_counts = (
    b_donors_unique
    .groupby(["condition_concept_id", "phecode"])["person_id"]
    .nunique()
    .rename("n_donors_B")
    .reset_index()
)


#STEP 3: build a_donors_unique from the dict ----------

# phenotype_dict: {(condition_concept_id, name): df, ...}

a_donors_unique = pd.concat(
    [
        df[["person_id"]]
        .drop_duplicates()
        .assign(condition_concept_id=cond_id)
        for (cond_id, name), df in filtered_cohort_dict.items()
    ],
    ignore_index=True,
)



# same as before: count donors per phenotype A
a_counts = (
    a_donors_unique
    .groupby("condition_concept_id")["person_id"]
    .nunique()
    .rename("n_donors_A")
    .reset_index()
)

# make merge keys the same dtype on both sides
for col in ["condition_concept_id", "person_id"]:
    a_donors_unique[col] = a_donors_unique[col].astype("Int64")
    b_donors_unique[col] = b_donors_unique[col].astype("Int64")

overlap_donors = a_donors_unique.merge(
    b_donors_unique,
    on=["condition_concept_id", "person_id"],
    how="inner"
)
# columns: condition_concept_id, donor_id, phecode

overlap_counts_donors = (
    overlap_donors
    .groupby(["condition_concept_id", "phecode"])["person_id"]
    .nunique()
    .rename("n_overlap_donors")
    .reset_index()
)



# start from max_overlap so we keep exactly those (A,B) pairs from File 3
result = max_overlap[["condition_concept_id", "phecode"]].drop_duplicates()

# add counts and overlaps
result = result.merge(a_counts, on="condition_concept_id", how="left")
result = result.merge(b_counts, on=["condition_concept_id", "phecode"], how="left")
result = result.merge(overlap_counts_donors, on=["condition_concept_id", "phecode"], how="left")

# fill missing counts/overlaps with 0
for col in ["n_donors_A", "n_donors_B", "n_overlap_donors"]:
    result[col] = result[col].fillna(0).astype(int)

# fractions (avoid division by zero)
result["frac_A"] = result["n_overlap_donors"] / result["n_donors_A"].replace(0, pd.NA)
result["frac_B"] = result["n_overlap_donors"] / result["n_donors_B"].replace(0, pd.NA)

# keep the columns you said you want
out = result[[
    "condition_concept_id",   # Phenotype A
    "phecode",                # Phenotype B
    "n_overlap_donors",       # Overlap
    "frac_A",                 # Phenotype A fraction
    "frac_B",                 # Phenotype B fraction
]]


result = result.merge(
    max_overlap[["condition_concept_id", "phecode", "frac_overlap"]],
    on=["condition_concept_id", "phecode"],
    how="left"
)



In [ ]:
result

In [ ]:
#function calls

'''
phe, phe_icd, phex, phex_icd = import_map_files()
phe_map = merge_phecodes_with_phenotype(phe, phe_icd)
phex_map = merge_phecodes_with_phenotype(phex, phex_icd)



cohort = get_data_pkl(data2, 'cohort_cond_df.pkl')
cohort_dict = get_cond_cohort_df_dict(cohort)
filtered_cohort_dict = filter_cond_df_dict(cohort_dict)
summary = get_icd_vocab_count_per_cohort(filtered_cohort_dict)
cohort_ICD_map = get_cohort_concept_id_to_ICD_mapping(cohort)
'''


In [ ]:
##visualize


#phe_map
#phex_map
#cohort
#filtered_cohort_dict
#summary
#flat_phe_map
#cohort_ICD_map

In [ ]:
filtered_cohort_dict